# 🚗 Australian Car Price Prediction

This project aims to predict used car prices in Australia using machine learning techniques. The goal is to analyze various vehicle attributes and build a predictive model capable of estimating car prices based on features such as make, model, year, mileage, fuel type, and transmission.

## 🎯 Objectives

- Analyze the Australian used car dataset.
- Identify key features influencing car prices.
- Train regression models for price prediction.
- Deploy an interactive web application for real-time predictions using **Streamlit**.

## 🧠 Technologies Used

- **Python**
- **Pandas**: For data manipulation and analysis.
- **Joblib**: For serialization and deserialization of models and transformers.
- **Streamlit**: For building the interactive web application interface.
- **Matplotlib / Seaborn**: For data visualization (if there are EDA sections in the notebook).
- **Scikit-learn / CatBoost / XGBoost**: For Machine Learning model development and training.

The final application allows users to input vehicle characteristics and obtain a predicted car price instantly through a web interface.

In [11]:
!pip3 install joblib
!pip3 install streamlit
!pip3 install pandas
!pip3 install catboost
!pip install category_encoders xgboost

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸Requirement already satisfied: category_encoders in /usr/local/lib/python3.12/dist-packages (2.9.0)


In [7]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from scipy.sparse import hstack, csr_matrix

@st.cache_resource
def load_assets():
    model = joblib.load('CarPredModel.pkl')
    te = joblib.load('target_encoder.pkl')
    ohe = joblib.load('one_hot_encoder.pkl')
    return model, te, ohe

model, te, ohe = load_assets()

brands = ['BMW', 'AUDI', 'HONDA', 'FORD', 'KIA', 'RENAULT', 'TOYOTA', 'SUZUKI', 'NISSAN',

'HOLDEN', 'SUBARU', 'VOLKSWAGEN', 'MERCEDES-BENZ', 'HYUNDAI', 'LAND-ROVER',

'MAZDA', 'LEXUS', 'MITSUBISHI', 'LDV', 'HAVAL', 'MINI', 'JEEP', 'CHEVROLET',

'ISUZU', 'VOLVO', 'MG', 'FIAT', 'JAGUAR', 'ŠKODA', 'ALFA-ROMEO', 'HSV', 'PORSCHE',

'DAIHATSU', 'SSANGYONG', 'CITROËN', 'DODGE', 'HINO', 'GWM', 'TESLA', 'IVECO',

'CHRYSLER', 'PEUGEOT']

models = ['SERIES 3', 'A6', 'ACCORD', 'RANGER', 'PICANTO', 'MASTER', 'COROLLA', 'SWIFT',

'NAVARA', 'COMMODORE', 'UTE', 'IMPREZA', 'RIO', 'POLO', 'S400H', 'I30',

'FAIRLANE', 'RANGE ROVER SPORT', 'RX-8', 'LANDCRUISER', 'KUGA', 'HILUX', 'IX35',

'RS3', 'CAMRY', 'FALCON', 'RODEO', 'OUTBACK', 'BT-50', 'RAPTOR', 'NX', 'CADDY',

'KLUGER', 'SONATA', 'DISCOVERY SPORT', 'SPORTAGE', 'DISCOVERY', 'EVEREST', 'E',

'GLS', 'TRITON', 'PULSAR', 'CRESSIDA', 'FOCUS', 'G10', 'ESTIMA', 'JUKE', 'FIESTA',

'PAJERO', 'JOLION', 'ACCENT', 'CIVIC', 'CR-V', 'RX', 'COOPER', 'ASTRA',

'GRAND CHEROKEE', 'CX-5', 'SANTA FE', 'GETZ', 'RAV4', 'EV6', 'SILVERADO', 'X4',

'YARIS', 'DMAX', 'M5', 'SCENIC', 'ASX', 'XC40', 'LANCER', 'D-MAX', 'TUCSON', 'NKR',

'ZS', 'CROWN', 'VITARA', 'A-CLASS', 'CRUZE', 'TIGUAN',

'XTRAIL ST 4WD AUTO REGO&RWC', 'ODYSSEY', 'MONDEO', 'CERATO', '370Z', 'PATROL',

'DUCATO', 'GOLF', 'A1', 'JETTA', 'SORENTO', 'MUSTANG', 'A5', 'A3', 'CX-9', 'X1',

'MURANO', 'COURIER', 'PASSAT', 'AMAROK', 'HIACE', 'GLADIATOR', 'F-150', 'TRAFIC',

'XV', 'CAPRICE', 'CAPTIVA', 'AURION', 'KANGOO', 'SERIES 1', 'MU-X', 'V60',

'SCIROCCO', 'CALAIS', 'C-HR', 'ELANTRA', 'WRANGLER', 'MONARO', 'LIBERTY', 'GLE',

'X-TYPE', 'HS', 'TERRITORY', 'COLORADO', 'KAROQ', 'GRAND VITARA', 'OPTIMA', 'ES',

'450SEL', 'OTHER', 'EOS', 'OUTLANDER', 'Q5', 'GIULIETTA', 'CHEROKEE', 'F-250',

'ILOAD', 'G6E', 'CLUB SPORT', 'FM450', 'CAYENNE', 'KONA', 'X5', 'X-TRAIL', 'I45',

'STARIA', 'DUALIS', '320I M SPORT', 'TRANSIT CUSTOM', 'ESCAPE', 'MIRAGE',

'FORESTER', '370GT', 'GS', 'QASHQAI', 'GLC', 'BEETLE', 'IS', 'PATHFINDER',

'RANGE ROVER', 'A4', 'CX-3', 'TIBURON', 'TRAX', 'CITY', 'XJ', 'CRAFTER',

'VELOSTER', 'EXPRESS', 'STATESMAN', 'X6', 'EXPLORER', 'TIIDA',

'HOLDEN KINGSWOOD', 'MACAN', 'VOLKSWAGEN 1600', 'FORD CAPRI', 'X3', 'PRELUDE',

'TOUAREG', 'CX-8', 'MERCEDES-BENZ C230', 'JAZZ', 'VITO', 'Z', 'ALPHARD',

'NISSAN SILVIA', 'COUNTRYMAN', 'BARINA', 'CLA', 'HOLDEN HQ', 'CX-7',

'DELIVER 9', 'FORTUNER', 'MINI', 'SX4', 'WRX', 'COMPASS', 'SQ5',

'MERCEDES-BENZ 250SE', 'FJ CRUISER', 'JIMNY', 'GT-R', 'RUKUS', 'MICRA', 'VIVA',

'MX-5', 'CORONA', 'IGNIS', 'MAGNA', 'E-CLASS', 'PRIUS', 'GTS', 'MEGANE', 'IMAX',

'STONIC', 'MAVERICK', 'CHALLENGER', 'HR-V', 'CAPTUR', 'SUPERB', 'CHARADE',

'GEMINI', 'DEFENDER', 'C-CLASS', 'XC60', 'S3', 'X7', 'I20', 'MERCEDES-BENZ X250',

'TRANSPORTER', 'SERIES 6', 'XF', 'OCTAVIA', 'D90', 'Q7', 'AVALON', 'MUSSO',

'EQUINOX', 'F-350', 'TRIBUTE', 'S-CROSS', 'ELGRAND', 'CELICA', 'TRUCK', 'Z4',

'TT', 'CLIO', 'CARNIVAL', 'MAXIMA', 'LS', 'FORD CORTINA', 'TRIBECA', 'C4',

'TOYOTA MARK II', 'T-CROSS', 'RANGE ROVER VELAR', 'TARAGO ULTIMA', 'AVENSIS',

'PATRIOT', 'SPRINTER', 'KOLEOS', 'FREEMONT', '5 SERIES', 'LEVORG', 'Q3',

'CORVETTE', 'YARIS CROSS', 'C63S', 'BALENO', 'H6', 'VIANO', 'NISSAN STAGEA',

'ECHO', 'FUSO', 'SERENA', 'V80', 'CRUZ', 'FAIRMONT', 'SKYLINE',

'RANGE ROVER EVOQUE', 'JCW', 'BERLINGO', 'S4', '320I E90', 'EXCEL',

'TOYOTA SOARER', 'BRAVO', '220I M SPORT', 'SUPRA', 'M3', 'Q2', '1 SERIES',

'MERCEDES-BENZ ML500', 'E46 325I', 'MERCEDES-BENZ ML250 CDI', '280CE',

'RONDO', '4RUNNER', 'LASER', 'SL', 'CANTER 515', 'E43 AMG', 'SELTOS', 'PALISADE',

'A7', 'COLT', 'B-CLASS', 'MERCEDES-BENZ 300 SEL', 'GTI', 'KOMBI', 'ALMERA',

'350GT', 'TRAILBLAZER', 'XC90', 'DUTRO', 'C43 AMG', 'INTEGRA',

'MERCEDES-BENZ ML250', 'TOYOTA AQUA', 'KIZASHI', 'MERCEDES-BENZ 220S',

'CRUISE', 'JOURNEY', 'TERIOS', 'DELICA', 'CANNON', '325I SERIES 3', 'ALTO',

'MODEL 3', 'PRADO', '200SX', 'CARAVELLE', 'NPR 300', 'SKYLINE 370GT',

'ECLIPSE CROSS', 'VENUE', '500X', 'DAILY', 'CUBE', 'SKYLINE 350GT', 'GLA',

'SERIES 2', 'SERIES 5', 'MULTIVAN', 'GRANDIS', 'S60', 'I40', 'M ROADSTER',

'LEGACY TOURING WAGON 2.0GT DIT', 'CIVIC TYPE R', 'MERCEDES-BENZ 220SE',

'BRZ', 'VALIANT', 'MR2', 'FRR500', 'BORA', 'MALIBU', 'GIULIA', 'STARLET', 'GLB',

'ECOSPORT', 'TOYOTA BB', 'TORANA', 'CAMIRA', 'FABIA',

'VOLKSWAGEN KARMANN GHIA', 'URVAN', 'PANDA', 'LEAF', 'REXTON',

'MERCEKANA', 'XE', 'S2000', '86 GTS', '350Z',

'TRANSIT', 'NQR', 'S5', 'T60', 'MERCEDES-BENZ ML320 CDI', 'FIT', 'FD',

'TARAGO GLX', 'NLR 45/150', 'HOLDEN ONE TONNER', '300 CE', 'STEED', 'KRUGER',

'NISSAN 180SX', 'SEDAN', 'M2', 'PAJERO SPORT', '190 E', 'CT', 'S-CLASS', 'V50',

'CC', 'A35 AMG', 'MERCEDES-BENZ ML350 CDI', 'A35', 'UX', 'AMG A35',

'E240 ELEGANCE', '560SEL', 'HOLDEN COUPE', 'INSIGHT', 'ACADIA', 'TOYOTA BLADE',

'4CYL AUTOMATIC', 'MERCEDES-BENZ G63 AMG SUV', 'C10', 'HOLDEN HJ',

'KIA PREGIO', 'MALOO', 'LAND ROVER SERIES III', 'TARAGO', 'BLUEBIRD',

'FIT HYBRID', 'A250 SPORT', 'S660', 'TERRACAN', '530I M SPORT', 'MODEL Y',

'TOYOTA', 'F-100', 'PT CRUISER', 'MG MGB', 'CREWMAN', 'PRIUS HYBRID',

'HOLDEN PREMIER', 'KODIAQ', 'RS 5', 'TARAGO GLI', 'CHRYSLER VALIANT CHARGER',

'CLUBMAN', 'V240', 'RCZ', 'DS', 'EH', 'CAMARO', 'CX-60', '118I M SPORT',

'325I COUPE', 'HOLDEN HZ', 'ZR-V', 'SERIES 4', 'S40', 'FESTIVA', 'CALIBER',

'TOYOTA VITZ', 'ACTROS', 'SR5', 'CX-30', 'PRIUS C', 'T-ROC', 'ACTYON', 'X2',

'SERIES 7', 'WITH REGO AND RWC', 'X TRAIL', 'SIRION', 'F-PACE', 'EVO',

'E46 325CI', 'YETI', '328I LUXURY LINE', 'ESCORT', 'ALTIMA', 'GENESIS',

'VELLFIRE', 'MERCEDES-BENZ V250', 'HOLDEN EJ', 'STINGER', 'TOYOTA CENTURY',

'TCROSS', 'RX7', 'VOXY']
fuel_types = ['DIESEL', 'PETROL', 'OTHER', 'HYBRID', 'ELECTRIC']
transmissions = ['AUTOMATIC', 'MANUAL']

st.set_page_config(layout="wide", page_title="Australian Car Price Prediction")

st.title('🚗 Australian Car Price Prediction')
st.markdown('### Insert the information about the car to get a prediction')

with st.sidebar:
    st.header('Car Features Input')
    ft_1 = st.selectbox('Brand', options=brands)
    ft_2 = st.selectbox('Model', options=models)
    ft_3 = st.number_input('Year', min_value=1950, max_value=2027, value=2018)
    ft_4 = st.number_input('Mileage (kms)', min_value=0, value=50000)
    ft_5 = st.selectbox('Fuel Type', options=fuel_types)
    ft_6 = st.selectbox('Transmission', options=transmissions)

state_default = 'NSW'

if st.button('Predict'):
    try:
        age = 2026 - ft_3

        data_raw = pd.DataFrame({
            'vehicle_model_display_name': [ft_2],
            'state': [state_default],
            'vehicle_fuel_type': [ft_5],
            'vehicle_transmission_type': [ft_6],
            'age': [age],
            'kms': [ft_4],
            'is_lux_brand': [1 if ft_1.lower() in ['mercedes-benz', 'bmw', 'audi'] else 0],
            'lux_score': [0],
            'pwr_score': [0],
            'util_score': [0]
        })

        X_te = te.transform(data_raw[['vehicle_model_display_name', 'state']])

        X_ohe = ohe.transform(data_raw[['vehicle_fuel_type', 'vehicle_transmission_type']])

        numeric_cols = ['age', 'kms', 'is_lux_brand', 'lux_score', 'pwr_score', 'util_score']
        X_num = csr_matrix(data_raw[numeric_cols].values)

        X_final = hstack([X_num, X_te, X_ohe])

        log_prediction = model.predict(X_final)
        real_prediction = np.expm1(log_prediction)

        st.success(f'### The predicted price for this car is: ${real_prediction[0]:,.2f}')

    except Exception as e:
        st.error(f"Error in processing: {e}. Check if column order matches the training.")

Writing app.py


## ▶️ Running the Application

To launch the web application locally, run the following command bellow:

```bash
streamlit run app.py

In [14]:
!streamlit run app.py

Clique aqui para abrir o site: NgrokTunnel: "https://pseudolinguistic-diffidently-alyse.ngrok-free.dev" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.53.22.145:8501

  Stopping...
  Stopping...
